# Eindhoven bicycle accidents filter and year grouping


In [3]:
import pandas as pd
import geopandas as gpd
from pathlib import Path
from shapely import wkt

In [4]:
# Set input and output paths
PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"
CLEAN_DIR = PROJECT_ROOT / "data" / "clean"

CLEAN_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PATH = RAW_DIR / "ongevallen_2022_2024.csv"
OUTPUT_PATH = CLEAN_DIR / "accidents_eindhoven_cycling.csv"
YEARLY_COUNTS_PATH = CLEAN_DIR / "accident_counts_by_year.csv"

# Load the raw accident dataset
df = pd.read_csv(INPUT_PATH, sep=",", low_memory=False)

print("Raw dataset shape:", df.shape)
print("Raw columns:")
print(df.columns.tolist())

# Check whether the required columns exist
required_columns = [
    "gemeente",
    "partij_1_objecttype",
    "partij_1_objecttype_overig",
    "partij_2_objecttype",
    "partij_2_objecttype_overig",
    "shape",
]

missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

# Filter records for cycling-related accidents in Eindhoven
bike_columns = [
    "partij_1_objecttype",
    "partij_1_objecttype_overig",
    "partij_2_objecttype",
    "partij_2_objecttype_overig",
]

location_mask = (
    df["gemeente"]
    .astype(str)
    .str.lower()
    .str.strip()
    .eq("eindhoven")
)

bike_mask = False

for col in bike_columns:
    bike_mask = bike_mask | df[col].astype(str).str.contains(
        "fiets|bicycle|bike",
        case=False,
        na=False,
        regex=True,
    )

df_accidents = df[location_mask & bike_mask].copy()

print("Filtered Eindhoven cycling accident records:", len(df_accidents))

Raw dataset shape: (382421, 40)
Raw columns:
['FID', 'verkeersongeval_nummer', 'jaar_ongeval', 'verkeersongeval_afloop', 'aantal_partijen', 'partij_1_objecttype', 'partij_1_objecttype_overig', 'partij_2_objecttype', 'partij_2_objecttype_overig', 'aard_ongeval', 'niveau_koppelen', 'wegsituatie', 'bebouwde_kom', 'maximum_snelheid', 'wegverlichting', 'wegverharding', 'wegdek', 'lichtgesteldheid', 'zichtafstand', 'weersgesteldheid', 'bijz_verkeersmaatregel', 'bijz_verkeersmaatregel_overig', 'bijz_infrastructuur', 'bijz_infrastructuur_overig', 'bijz_tijdelijke_aard', 'bijz_tijdelijke_aard_overig', 'junctie_id', 'wegvak_id', 'hectometer', 'straatnaam', 'woonplaats', 'actueel', 'wegbeheerder', 'gemeente', 'provincie', 'dienstnaam', 'districtnaam', 'indicatie_alcohol', 'gdb_geomattr_data', 'shape']
Filtered Eindhoven cycling accident records: 1653


In [5]:
# Rename Dutch column names to English column names
rename_map = {
    "FID": "fid",
    "verkeersongeval_nummer": "accident_id",
    "jaar_ongeval": "accident_year",
    "verkeersongeval_afloop": "accident_outcome",
    "aantal_partijen": "number_of_parties",
    "partij_1_objecttype": "party_1_object_type",
    "partij_1_objecttype_overig": "party_1_object_type_other",
    "partij_2_objecttype": "party_2_object_type",
    "partij_2_objecttype_overig": "party_2_object_type_other",
    "aard_ongeval": "accident_nature",
    "niveau_koppelen": "linking_level",
    "wegsituatie": "road_situation",
    "bebouwde_kom": "built_up_area",
    "maximum_snelheid": "speed_limit",
    "wegverlichting": "road_lighting",
    "wegverharding": "road_pavement",
    "wegdek": "road_surface",
    "lichtgesteldheid": "light_condition",
    "zichtafstand": "visibility_distance",
    "weersgesteldheid": "weather_condition",
    "bijz_verkeersmaatregel": "special_traffic_measure",
    "bijz_verkeersmaatregel_overig": "special_traffic_measure_other",
    "bijz_infrastructuur": "special_infrastructure",
    "bijz_infrastructuur_overig": "special_infrastructure_other",
    "bijz_tijdelijke_aard": "special_temporary_nature",
    "bijz_tijdelijke_aard_overig": "special_temporary_nature_other",
    "junctie_id": "junction_id",
    "wegvak_id": "road_section_id",
    "hectometer": "hectometer",
    "straatnaam": "street_name",
    "woonplaats": "place_name",
    "actueel": "current",
    "wegbeheerder": "road_authority",
    "gemeente": "municipality",
    "provincie": "province",
    "dienstnaam": "service_area_name",
    "districtnaam": "district_name",
    "indicatie_alcohol": "alcohol_indication",
    "gdb_geomattr_data": "gdb_geometry_attribute_data",
    "shape": "geometry",
}

existing_rename_map = {
    old_name: new_name
    for old_name, new_name in rename_map.items()
    if old_name in df_accidents.columns
}

df_accidents = df_accidents.rename(columns=existing_rename_map)

print("Renamed columns:")
print(df_accidents.columns.tolist())

# Standardize accident outcome values
if "accident_outcome" in df_accidents.columns:
    print("Raw accident outcome values:")
    print(df_accidents["accident_outcome"].value_counts(dropna=False))

    outcome_map = {
        "Uitsluitend materiele schade": "property_damage_only",
        "Uitsluitend materiële schade": "property_damage_only",
        "Letsel": "injury",
        "Dodelijk": "fatal",
    }

    df_accidents["accident_outcome_standardized"] = (
        df_accidents["accident_outcome"]
        .map(outcome_map)
        .fillna(df_accidents["accident_outcome"])
    )

# Convert the current-status field from Ja/Nee to boolean values
if "current" in df_accidents.columns:
    current_map = {
        "Ja": True,
        "Nee": False,
        "ja": True,
        "nee": False,
    }

    df_accidents["current_boolean"] = df_accidents["current"].map(current_map)

# Parse WKT geometry values
def parse_wkt_geometry(value):
    if pd.isna(value):
        return None

    try:
        return wkt.loads(str(value))
    except Exception:
        return None

df_accidents["geometry_object"] = df_accidents["geometry"].apply(parse_wkt_geometry)

invalid_geometry_count = df_accidents["geometry_object"].isna().sum()
print("Invalid or missing geometry rows:", invalid_geometry_count)

Renamed columns:
['fid', 'accident_id', 'accident_year', 'accident_outcome', 'number_of_parties', 'party_1_object_type', 'party_1_object_type_other', 'party_2_object_type', 'party_2_object_type_other', 'accident_nature', 'linking_level', 'road_situation', 'built_up_area', 'speed_limit', 'road_lighting', 'road_pavement', 'road_surface', 'light_condition', 'visibility_distance', 'weather_condition', 'special_traffic_measure', 'special_traffic_measure_other', 'special_infrastructure', 'special_infrastructure_other', 'special_temporary_nature', 'special_temporary_nature_other', 'junction_id', 'road_section_id', 'hectometer', 'street_name', 'place_name', 'current', 'road_authority', 'municipality', 'province', 'service_area_name', 'district_name', 'alcohol_indication', 'gdb_geometry_attribute_data', 'geometry']
Raw accident outcome values:
accident_outcome
Uitsluitend materiele schade    1301
Letsel                           348
Dodelijk                           4
Name: count, dtype: int64

In [6]:
df_accidents = df_accidents.dropna(subset=["geometry_object"]).copy()

# Convert RD New coordinates to WGS84 longitude and latitude
accidents_gdf = gpd.GeoDataFrame(
    df_accidents,
    geometry="geometry_object",
    crs="EPSG:28992",
)

accidents_gdf = accidents_gdf.to_crs("EPSG:4326")

accidents_gdf["longitude"] = accidents_gdf.geometry.x
accidents_gdf["latitude"] = accidents_gdf.geometry.y

print("Coordinate sample:")
print(accidents_gdf[["longitude", "latitude"]].head())

# Select the columns needed for the cleaned accident dataset
clean_columns = [
    "fid",
    "accident_id",
    "accident_year",
    "accident_outcome",
    "accident_outcome_standardized",
    "number_of_parties",
    "party_1_object_type",
    "party_1_object_type_other",
    "party_2_object_type",
    "party_2_object_type_other",
    "accident_nature",
    "linking_level",
    "road_situation",
    "built_up_area",
    "speed_limit",
    "road_lighting",
    "road_pavement",
    "road_surface",
    "light_condition",
    "visibility_distance",
    "weather_condition",
    "special_traffic_measure",
    "special_traffic_measure_other",
    "special_infrastructure",
    "special_infrastructure_other",
    "special_temporary_nature",
    "special_temporary_nature_other",
    "junction_id",
    "road_section_id",
    "hectometer",
    "street_name",
    "place_name",
    "current",
    "current_boolean",
    "road_authority",
    "municipality",
    "province",
    "service_area_name",
    "district_name",
    "alcohol_indication",
    "gdb_geometry_attribute_data",
    "geometry",
    "longitude",
    "latitude",
]

clean_columns = [
    col for col in clean_columns
    if col in accidents_gdf.columns
]

df_accidents_clean = pd.DataFrame(accidents_gdf[clean_columns]).copy()

# Save the cleaned accident dataset
# df_accidents_clean.to_csv(OUTPUT_PATH, index=False)

# print("Saved clean accident dataset to:")
# print(OUTPUT_PATH)
print("Clean dataset shape:", df_accidents_clean.shape)

# Count cycling-related accidents by year
accident_counts_by_year = (
    df_accidents_clean
    .groupby("accident_year")
    .size()
    .reset_index(name="accident_count")
    .sort_values("accident_year")
)

print("Accident counts by year:")
print(accident_counts_by_year.to_string(index=False))

df_accidents_clean.head(20)
accident_counts_by_year.to_csv(YEARLY_COUNTS_PATH, index=False)

Coordinate sample:
      longitude   latitude
277    5.396512  51.436566
708    5.472431  51.434861
968    5.475478  51.441108
970    5.514564  51.441977
1383   5.481209  51.444889
Clean dataset shape: (1653, 44)
Accident counts by year:
 accident_year  accident_count
          2022             545
          2023             551
          2024             557
